# BUILDING THE DATAFRAME
This Python Notebook shows the operations of cleaning data, pivoting columns, and adding relevant information from different sources.

## STEP 1. Calling the database from ISTAT's API
To circumvent ISTAT's request limitations, the code saves a cached version of the data on the local drive and checks its date before calling the API again.

In [34]:
import requests
import time
from pathlib import Path

url = "https://esploradati.istat.it/SDMXWS/rest/data/41_983"
headers = {"Accept": "application/vnd.sdmx.data+csv;version=1.0.0"}
path = Path("../data/0_rawdata.csv")
maxage = 24 * 7
download_needed = True

# Check if file exists and if it was modified less than a week ago
if path.exists():
    cacheage = (time.time() - path.stat().st_mtime) / 3600
    if cacheage < maxage:
        download_needed = False
# Checks if download_needed is still True and downloads
if download_needed:
    r = requests.get(url, headers=headers, verify=False)
    path.write_bytes(r.content)
    print("File downloaded.")
else:
    print("Using cached file.")

Using cached file.


## STEP 2. Load data and clean up columns
As observed in the first notebook, most columns are not needed: DATAFLOW and FREQ hold the same value in every row, whereas OBS_STATUS, NOTE_DS, NOTE_REF_AREA, NOTE_DATA_TYPE, NOTE_RESULT, NOTE_TIME_PERIOD, BASE_PER, UNIT_MEAS, UNIT_MULT are all empty.

In [35]:
import pandas as pd

df = pd.read_csv(path, dtype=str)
df = df.drop(columns=["DATAFLOW", "FREQ", "OBS_STATUS", "NOTE_DS", "NOTE_REF_AREA", "NOTE_DATA_TYPE", "NOTE_RESULT", "NOTE_TIME_PERIOD", "BASE_PER", "UNIT_MEAS", "UNIT_MULT"])
df.head()

,REF_AREA,DATA_TYPE,RESULT,TIME_PERIOD,OBS_VALUE
0,001001,KILLINJ,F,2001,10
1,001001,KILLINJ,F,2002,10
2,001001,KILLINJ,F,2003,7
3,001001,KILLINJ,F,2004,13
4,001001,KILLINJ,F,2005,2


We can now merge the DATA_TYPE and RESULT columns into a single column, named OBSERVATION.

In [36]:
# Create a new column "OBSERVATION"
df["OBSERVATION"] = None

# When DATA_TYPE is ROADACC and RESULT is 9, the OBSERVATION column should be ACCIDENTS
df.loc[(df["DATA_TYPE"] == "ROADACC") & (df["RESULT"] == "9"), "OBSERVATION"] = "ACCIDENTS"
# When DATA_TYPE is KILLINJ and RESULT is F, the OBSERVATION column should be INJURIES
df.loc[(df["DATA_TYPE"] == "KILLINJ") & (df["RESULT"] == "F"), "OBSERVATION"] = "INJURIES"
# When DATA_TYPE is KILLINJ and RESULT is M, the OBSERVATION column should be DEATHS
df.loc[(df["DATA_TYPE"] == "KILLINJ") & (df["RESULT"] == "M"), "OBSERVATION"] = "DEATHS"

# Remove DATA_TYPE and RESULT columns, as they are not needed anymore
df = df.drop(columns=["DATA_TYPE", "RESULT"])
# Move the OBSERVATION column
df = df[["REF_AREA", "TIME_PERIOD", "OBSERVATION", "OBS_VALUE"]]

# Convert OBS_VALUE to integer
df[["OBS_VALUE"]] = df[["OBS_VALUE"]].astype(int)

df.head()

,REF_AREA,TIME_PERIOD,OBSERVATION,OBS_VALUE
0,001001,2001,INJURIES,10
1,001001,2002,INJURIES,10
2,001001,2003,INJURIES,7
3,001001,2004,INJURIES,13
4,001001,2005,INJURIES,2


Instead of having single rows foraccidents, injuries and deaths for each location and year, a single conbination of location and year can have all the values for injuries, deaths and accidents through pivoting.

In [37]:
istat = (df.pivot_table(index=["REF_AREA", "TIME_PERIOD"], columns="OBSERVATION", values="OBS_VALUE", aggfunc="sum").reset_index())
istat.columns.name = None
istat.head()

,REF_AREA,TIME_PERIOD,ACCIDENTS,DEATHS,INJURIES
0,001001,2001,5,0,10
1,001001,2002,5,0,10
2,001001,2003,4,0,7
3,001001,2004,9,0,13
4,001001,2005,2,0,2


## STEP 3. Integration with SITUAS
Since we don't have the location names but just the ISTAT codes, we need to cross-reference the data with another database. I downloaded the SITUAS data from 31 December 2024 as "situas.csv" through the built-in download function.

In [38]:
situasfile = Path("../data/1_situas.csv")
situas_raw = pd.read_csv(situasfile, sep=";", dtype=str)

print(situas_raw.shape)
situas_raw.head()

(7896, 17)


,Codice Ripartizione geografica,Codice Regione,Codice Provincia (Storico),Codice Provincia/Uts,Codice Comune (alfanumerico),Codice Comune (numerico),Comune,Comune (dizione straniera),Sigla automobilistica,Capoluogo di Provincia/Uts,Capoluogo di Regione,Popolazione legale,Anno Censimento,Superficie (Kmq),Anno (Superficie),Popolazione residente,Anno (Popolazione residente)
0,1,01,001,201,001001,1001,Agliè,NaN,TO,0,0,2562,2021,"13,1463",2024,2585,2024
1,1,01,001,201,001002,1002,Airasca,NaN,TO,0,0,3660,2021,"15,7393",2024,3695,2024
2,1,01,001,201,001003,1003,Ala di Stura,NaN,TO,0,0,467,2021,"46,3316",2024,463,2024
3,1,01,001,201,001004,1004,Albiano d'Ivrea,NaN,TO,0,0,1637,2021,"11,7397",2024,1624,2024
4,1,01,001,201,001006,1006,Almese,NaN,TO,0,0,6331,2021,"17,8741",2024,6297,2024


From the resulting dataframe, I only want to keep a few columns that might be relevant: "Codice Comune (alfanumerico)", "Comune", "Sigla automobilistica", "Capoluogo di Provincia/Uts", "Capoluogo di Regione", "Superficie (Kmq)", "Popolazione residente". I will rename them in english and convert values that are meant to be numeric into numbers.

In [39]:
# Function that removes thousand periods and replaces decimal commas with decimal periods
def to_number(num):
    return pd.to_numeric(num.str.replace(".", "", regex=False).str.replace(",", ".", regex=False))

situas = situas_raw[[
    "Codice Comune (alfanumerico)",
    "Comune",
    "Sigla automobilistica",
    "Capoluogo di Provincia/Uts",
    "Capoluogo di Regione",
    "Superficie (Kmq)",
    "Popolazione residente",
]].copy()

situas.columns = [
    "REF_AREA",
    "NAME",
    "PROVINCE",
    "IS_PROVINCE_CAPITAL",
    "IS_REGION_CAPITAL",
    "AREA",
    "POPULATION",
]

# Converts area value to number
situas["AREA"] = to_number(situas["AREA"])
# Converts popilation value to number
situas["POPULATION"] = to_number(situas["POPULATION"]).astype(int)
# Sets values that are suupposed to be boolean to actual True and False values
situas["IS_PROVINCE_CAPITAL"] = situas["IS_PROVINCE_CAPITAL"] == "1"
situas["IS_REGION_CAPITAL"] = situas["IS_REGION_CAPITAL"] == "1"

print(situas.shape)
situas.head()

(7896, 7)


,REF_AREA,NAME,PROVINCE,IS_PROVINCE_CAPITAL,IS_REGION_CAPITAL,AREA,POPULATION
0,001001,Agliè,TO,False,False,13.1463,2585
1,001002,Airasca,TO,False,False,15.7393,3695
2,001003,Ala di Stura,TO,False,False,46.3316,463
3,001004,Albiano d'Ivrea,TO,False,False,11.7397,1624
4,001006,Almese,TO,False,False,17.8741,6297


Before merging the two dataframes, it sould be noted that, in 2024, municipalieties with no accidents have been omitted from the ISTAT file. Thus, it's necessary to add the municipalietes that are present in SITUAS but not in ISTAT, filling values with zero, otherwise statistics based on 2024 would be wrong.

In [40]:
# Select all the municipalities that appear in 2024
present_last = set(istat.loc[istat["TIME_PERIOD"] == 2024, "REF_AREA"])
# Select the municipalities that appear in SITUAS but not in ISTAT in 2024
missing_last = sorted(set(situas["REF_AREA"]) - present_last) 

# Creates a nnew table with the missing municipalities and adds zeros to their accidents, deaths and injuries values
zeros = pd.DataFrame({
    "REF_AREA": missing_last,
    "TIME_PERIOD": 2024,
    "ACCIDENTS": 0,
    "DEATHS": 0,
    "INJURIES": 0,
})

# Adds the rows from the new table at the end of the original istat dataframe
istat = pd.concat([istat, zeros], ignore_index=True)

print(istat.shape)

(199080, 5)


Now, we can join the istat and situas tables on the municipality code and calculate the rates by 1000 inhabitants and by square kilometer.

**Note:** only municipalities present in the SITUAS tble will be kept: there might be municipalities present in the ISTAT table that have been suppressed or merged in 2024, but those are not useful for statistics and are probably smaller municipalities, which hold a small weight on total statistics.

In [41]:
# Perform an inner merge
final = istat.merge(situas, on="REF_AREA", how="inner")

# Calculate accidents, deaths and injuries per 1000 inhabitants
final["ACCIDENTS_PER_1000_INH"] = final["ACCIDENTS"] / final["POPULATION"] * 1000
final["DEATHS_PER_1000_INH"] = final["DEATHS"] / final["POPULATION"] * 1000
final["INJURIES_PER_1000_INH"] = final["INJURIES"] / final["POPULATION"] * 1000

# Calculate accidents, deaths and injuries per square kilometer
final["ACCIDENTS_PER_KMQ"] = final["ACCIDENTS"] / final["AREA"]
final["DEATHS_PER_KMQ"] = final["DEATHS"] / final["AREA"]
final["INJURIES_PER_KMQ"] = final["INJURIES"] / final["AREA"]

# Order result by municipality code
final = final.sort_values(["REF_AREA", "TIME_PERIOD"]).reset_index(drop=True)

print(final.shape)
final.head()

(190465, 17)


,REF_AREA,TIME_PERIOD,ACCIDENTS,DEATHS,INJURIES,NAME,PROVINCE,IS_PROVINCE_CAPITAL,IS_REGION_CAPITAL,AREA,POPULATION,ACCIDENTS_PER_1000_INH,DEATHS_PER_1000_INH,INJURIES_PER_1000_INH,ACCIDENTS_PER_KMQ,DEATHS_PER_KMQ,INJURIES_PER_KMQ
0,001001,2024,0,0,0,Agliè,TO,False,False,13.1463,2585,0.000000,0.0,0.000000,0.000000,0.0,0.000000
1,001001,2001,5,0,10,Agliè,TO,False,False,13.1463,2585,1.934236,0.0,3.868472,0.380335,0.0,0.760670
2,001001,2002,5,0,10,Agliè,TO,False,False,13.1463,2585,1.934236,0.0,3.868472,0.380335,0.0,0.760670
3,001001,2003,4,0,7,Agliè,TO,False,False,13.1463,2585,1.547389,0.0,2.707930,0.304268,0.0,0.532469
4,001001,2004,9,0,13,Agliè,TO,False,False,13.1463,2585,3.481625,0.0,5.029014,0.684603,0.0,0.988871


## STEP 3. Export the resulting dataframe
The resulting dataframe can be exported to .csv, so that it can be used for statistics and metrics.

In [42]:
output = Path("../data/2_final.csv")
final.to_csv(output, index=False)

print(f"Saved file.")

Saved file.


## STEP 4. Check missing values
There might be missing values caused by exporting or joining.

In [43]:
import pandas as pd
from pathlib import Path

finalpath = Path("../data/2_final.csv")
df = pd.read_csv(finalpath)

# Make the REF:AREA a 6-digit code with leading zeros
df["REF_AREA"] = df["REF_AREA"].astype(str).str.zfill(6)

print(df.shape)
pd.DataFrame({"dtype": df.dtypes, "missing": df.isna().sum(), "distinct": df.nunique()})

(190465, 17)


,dtype,missing,distinct
REF_AREA,str,0,7896
TIME_PERIOD,int64,0,24
ACCIDENTS,int64,0,1223
DEATHS,int64,0,89
INJURIES,int64,0,1471
NAME,str,25,7890
PROVINCE,str,2299,106
IS_PROVINCE_CAPITAL,bool,0,2
IS_REGION_CAPITAL,bool,0,2
AREA,float64,0,7860


There appear to be 25 rows with no "NAME" attribute. All of them refer to the same REF_AREA value.

In [44]:
# Filter rows where NAME is missing
unmatched = df[df["NAME"].isna()]
print(unmatched["REF_AREA"].unique())

<StringArray>
['001168']
Length: 1, dtype: str


This REF_AREA value refers to the ISTAT code of the [Italian city of None](https://it.wikipedia.org/wiki/None_(Italia)), near Turin. Due to its name, it's possible that the value was filtered out when cleaning empty values (often referred to as "None").

In [45]:
# Putting back the value in NAME
df.loc[df["REF_AREA"] == "001168", "NAME"] = "None"

# Check number of rows where "NAME" is empty, it should be 0
print(len(df[df["NAME"].isna()]))

0


As for the rows with a missing PROVINCE value, by printing the unique values it's obvious that the value was supposed to be "NA", the province of Naples, which is yet another way to indicate empty values like None.

In [46]:
# Filter rows where PROVINCE is missing
missing_prov = df[df['PROVINCE'].isna()]
print(missing_prov['NAME'].unique())

<StringArray>
[                   'Acerra',                  'Afragola',
                   'Agerola',                  'Anacapri',
                    'Arzano',                    'Bacoli',
           'Barano d'Ischia',                'Boscoreale',
              'Boscotrecase',                 'Brusciano',
                   'Caivano',                'Calvizzano',
                 'Camposano',                     'Capri',
         'Carbonara di Nola',                   'Cardito',
      'Casalnuovo di Napoli',              'Casamarciano',
        'Casamicciola Terme',                'Casandrino',
                'Casavatore',          'Casola di Napoli',
                   'Casoria',   'Castellammare di Stabia',
      'Castello di Cisterna',                   'Cercola',
                  'Cicciano',                  'Cimitile',
                 'Comiziano',                  'Crispano',
                     'Forio',            'Frattamaggiore',
              'Frattaminore',     'Giuglia

Again, in order to have a complee dataset, the province has to be filled with the coresponding value.

In [47]:
# Putting back the value in PROVINCE
df.loc[df["PROVINCE"].isna(), "PROVINCE"] = "NA"

# Check number of rows where "NAME" is empty, it should be 0
print(len(df[df["PROVINCE"].isna()]))

0


The complete dataset can now be saved again.

In [48]:
df.to_csv(output, index=False)

print(f"Saved file.")

Saved file.
